In [1]:
import torch
import tiktoken
from tiktoken.load import load_tiktoken_bpe
import os 
from pathlib import Path

In [2]:
def text_to_token_ids(text,tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)

In [3]:
def token_ids_to_text(ids,tokenizer):
    return tokenizer.decode(ids.squeeze(0).tolist())

In [4]:
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096,freq_config=None):
    assert head_dim % 2 == 0, "Embedding dimension must be even"
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
    
    if freq_config is not None:
        low_freq_wavelen = freq_config["original_context_length"] / freq_config["low_freq_factor"]
        high_freq_wavelen = freq_config["original_context_length"] / freq_config["high_freq_factor"]
        wavelen = 2 * torch.pi / inv_freq
        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen, inv_freq / freq_config["factor"], inv_freq
        )

        smooth_factor = (freq_config["original_context_length"] / wavelen - freq_config["low_freq_factor"]) / (
            freq_config["high_freq_factor"] - freq_config["low_freq_factor"]
        )

        smoothed_inv_freq = (
            (1 - smooth_factor) * (inv_freq / freq_config["factor"]) + smooth_factor * inv_freq
        )
        is_medium_freq = (wavelen <= low_freq_wavelen) & (wavelen >= high_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
        inv_freq = inv_freq_llama
    
    
    positions = torch.arange(context_length)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  
    angles = torch.cat([angles, angles], dim=1)  
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

In [5]:
def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"
    x1 = x[..., : head_dim // 2] 
    x2 = x[..., head_dim // 2 :]  
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)
    return x_rotated.to(dtype=x.dtype)

In [6]:
def generate_text(model,idx,max_new_tokens,context_size,temperature=0.0,top_k=None,eos_id=None):
    
    for _ in range(max_new_tokens):
        idx_cond=idx[:,-context_size:]
        with torch.no_grad():
            logits=model(idx_cond)
        logits=logits[:,-1,:]
        #probas=torch.softmax(next_logits,dim=-1)
        if top_k is not None:
            top_k_logits,_=torch.topk(logits,top_k,dim=-1)
            min_val=top_k_logits[:,-1]
            logits=torch.where(logits < min_val,torch.tensor(float('-inf')).to(logits.device),logits)
        if temperature > 0.0:
            logits=logits/temperature
            probas=torch.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probas,num_samples=1)
        else :
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and torch.all(idx_next == eos_id):
            break
        idx=torch.cat((idx,idx_next),dim=-1)
    return idx

In [7]:
def calc_loss_batch(inputs,target,model,device):
    inputs=inputs.to(device)
    target=target.to(device)
    with torch.inference_mode():
        logits=model(inputs)
    return torch.nn.functional.cross_entropy(logits.flatten(0,1),target.flatten())

def calc_loss_loader(dataloader,model,device,num_batches=None):
    total_loss=0
    if(len(dataloader)==0):return float("nan")
    elif(num_batches==None):
        num_batches=len(dataloader)
    else:
        num_batches=min(num_batches,len(dataloader))

    for i,(inp_batch,tar_batch) in enumerate(dataloader):
        if(i >= num_batches):break
        total_loss+=calc_loss_batch(inp_batch,tar_batch,model,device).item()
    
    return total_loss/num_batches

In [8]:
def evaluate(model,train_loader,val_loader,eval_it,device):
    model.eval()
    train_loss=calc_loss_loader(train_loader,model,device,eval_it)
    val_loss=calc_loss_loader(val_loader,model,device,eval_it)
    model.train()
    return train_loss,val_loss

In [9]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.cfg["context_length"]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )
    print(token_ids_to_text(token_ids, tokenizer))
    model.train()

In [10]:
def assign(left, right, tensor_name="unknown"):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")
    
    with torch.no_grad():
        if isinstance(right, torch.Tensor):
            left.copy_(right)
        else:
            left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

    return left 

In [11]:
def load_weights_into_llama(model, param_config, params):
    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):

        # Load attention weights
        model.trf_blocks[l].att.W_query.weight = assign(
            model.trf_blocks[l].att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        model.trf_blocks[l].att.W_key.weight = assign(
            model.trf_blocks[l].att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        model.trf_blocks[l].att.W_value.weight = assign(
            model.trf_blocks[l].att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )
        model.trf_blocks[l].att.out_proj.weight = assign(
            model.trf_blocks[l].att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )
        model.trf_blocks[l].norm1.weight = assign(
            model.trf_blocks[l].norm1.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # Load FeedForward weights
        model.trf_blocks[l].ff.fc1.weight = assign(
            model.trf_blocks[l].ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        model.trf_blocks[l].ff.fc2.weight = assign(
            model.trf_blocks[l].ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        model.trf_blocks[l].ff.fc3.weight = assign(
            model.trf_blocks[l].ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        model.trf_blocks[l].norm2.weight = assign(
            model.trf_blocks[l].norm2.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Load output layer weights
    model.final_norm.weight = assign(model.final_norm.weight, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params.keys():
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")




In [12]:
class GroupedQueryAttention(torch.nn.Module):
    def __init__(self,d_in,d_out,num_heads,num_kv_groups,dtype=None):
        super().__init__()
        assert d_out%num_heads==0,"Output dimension must be divisible by number of heads"
        assert num_heads%num_kv_groups==0,"Number of heads must be divisible by number of kv groups"
        self.d_out=d_out
        self.kv_groups=num_kv_groups
        self.num_heads=num_heads
        self.head_dim=d_out//num_heads
        self.group_size = num_heads // num_kv_groups
        self.W_query=torch.nn.Linear(d_in,d_out,bias=False,dtype=dtype)
        self.W_key=torch.nn.Linear(d_in,self.kv_groups*self.head_dim,bias=False,dtype=dtype)
        self.W_value=torch.nn.Linear(d_in,self.kv_groups*self.head_dim,bias=False,dtype=dtype)
        self.out_proj=torch.nn.Linear(d_out,d_out,bias=False,dtype=dtype)
    def forward(self,x,mask=None,cos=None,sin=None):
        b,num_tokens,d_in=x.shape
        queries=self.W_query(x)
        keys=self.W_key(x)
        values=self.W_value(x)#(b,num_tokens,d_out)
        queries=queries.reshape(b,num_tokens,self.num_heads,self.head_dim)
        keys=keys.reshape(b,num_tokens,self.kv_groups,self.head_dim)
        values=values.reshape(b,num_tokens,self.kv_groups,self.head_dim)
        queries=queries.transpose(1,2)#(b,num_heads,num_tokens,head_dim)
        keys=keys.transpose(1,2)
        values=values.transpose(1,2)
        
        if cos is not None:
            keys=compute_rope(keys,cos,sin)
            queries=compute_rope(queries,cos,sin)
        keys=keys.repeat_interleave(self.group_size,dim=1)
        values=values.repeat_interleave(self.group_size,dim=1)
        attn_scores=queries@keys.transpose(2,3)
        if mask is None:
            mask=torch.triu(torch.ones(num_tokens,num_tokens),diagonal=1)

        attn_scores=attn_scores.masked_fill(mask,-torch.inf)
        attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)#(b,num_heads,num_tokens,head_dim)
        context_vec=(attn_weights@values).transpose(1,2)
        context_vec=context_vec.reshape(b,num_tokens,self.d_out)
        context_vec=self.out_proj(context_vec)
        return context_vec

In [13]:
class RMSNorm(torch.nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps=1e-5
        self.weight=torch.nn.Parameter(torch.ones(emb_dim))
    def forward(self,x):
        sq_mean=x.pow(2).mean(dim=-1,keepdim=True)
        x_norm=self.weight*x*torch.rsqrt(self.eps+sq_mean)
        return x_norm.to(dtype=x.dtype)

In [14]:
class SiLU(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        return x*torch.sigmoid(x)

In [15]:
class FeedForward(torch.nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.fc1=torch.nn.Linear(cfg["emb_dim"],cfg["hidden_dim"],dtype=cfg["dtype"],bias=False)
        self.fc2=torch.nn.Linear(cfg["emb_dim"],cfg["hidden_dim"],dtype=cfg["dtype"],bias=False)
        self.fc3=torch.nn.Linear(cfg["hidden_dim"],cfg["emb_dim"],dtype=cfg["dtype"],bias=False)
        self.silu=SiLU()
    def forward(self,x):
        x_fc1=self.fc1(x)
        x_fc2=self.fc2(x)
        x=self.silu(x_fc1)*x_fc2
        x=self.fc3(x)
        return x

In [16]:
class Transformer(torch.nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.att=GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            dtype=cfg["dtype"]
        )
        self.norm1=RMSNorm(cfg["emb_dim"])
        self.norm2=RMSNorm(cfg["emb_dim"])
        self.ff=FeedForward(cfg)

    def forward(self,x,mask=None,cos=None,sin=None):
        shortcut=x
        x=self.norm1(x)
        x = self.att(x.to(torch.bfloat16), mask, cos, sin)
        x=x+shortcut

        shortcut=x
        x=self.norm2(x)
        x = self.ff(x.to(torch.bfloat16))
        x=x+shortcut
        return x

In [17]:
class LLama3Model(torch.nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=torch.nn.Embedding(cfg["vocab_size"],cfg["emb_dim"],dtype=cfg["dtype"])
        self.trf_blocks=torch.nn.Sequential(*(Transformer(cfg) for _ in range(cfg["n_layers"])))
        self.final_norm=RMSNorm(cfg["emb_dim"])
        self.out_head=torch.nn.Linear(cfg["emb_dim"],cfg["vocab_size"],bias=False,dtype=cfg["dtype"])
        cos, sin = precompute_rope_params(
            head_dim=cfg["emb_dim"] // cfg["n_heads"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            freq_config=cfg["rope_freq"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg=cfg
    def forward(self,in_idx):
        batch_sz,seq_len=in_idx.shape
        x=self.tok_emb(in_idx)
        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x=self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

In [18]:
class Tokenizer:
    def __init__(self, model_path):
        if not os.path.isfile(model_path):
            raise FileNotFoundError(model_path)

        mergeable = load_tiktoken_bpe(model_path)
        
        self.special = {
            "<|begin_of_text|>": 128000,
            "<|end_of_text|>": 128001,
            "<|start_header_id|>": 128006,
            "<|end_header_id|>": 128007,
            "<|eot_id|>": 128009,
        }
        self.bos_id = self.special["<|begin_of_text|>"]
        self.eos_id = self.special["<|end_of_text|>"]
        self.eot_id = self.special["<|eot_id|>"]
        
        self.special.update({f"<|reserved_{i}|>": 128002 + i
                             for i in range(256)
                             if 128002 + i not in self.special.values()})

        self.model = tiktoken.Encoding(
            name=Path(model_path).name,
            pat_str=r"(?i:'s|'t|'re|'ve|'m|'ll|'d)"
                    r"|[^\r\n\p{L}\p{N}]?\p{L}+"
                    r"|\p{N}{1,3}"
                    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
                    r"|\s*[\r\n]+"
                    r"|\s+(?!\S)"
                    r"|\s+",
            mergeable_ranks=mergeable,
            special_tokens=self.special,
        )

    def encode(self, text, bos=False, eos=False):
        ids = ([self.special["<|begin_of_text|>"]] if bos else []) \
              + self.model.encode(text,allowed_special="all")
        if eos:
            ids.append(self.special["<|end_of_text|>"])
        return ids

    def decode(self, ids):
        return self.model.decode(ids)

In [19]:
LLAMA3_CONFIG_1B = {
    "vocab_size": 128_256,
    "context_length": 8192,
    "emb_dim": 2048,
    "n_heads": 32,
    "n_layers": 16,
    "hidden_dim": 8192,
    "n_kv_groups": 8,
    "rope_base": 500_000.0,
    "rope_freq": None,
    "dtype": torch.bfloat16
}

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
from safetensors.torch import load_file
tokenizer_file = r"C:\Users\Binish\Desktop\LLM_FROM_SCRATCH\models\Llama-3.2-1B-Instruct\original\tokenizer.model"
combined_weights = load_file(r"C:\Users\Binish\Desktop\LLM_FROM_SCRATCH\models\Llama-3.2-1B-Instruct\model.safetensors")

In [22]:
import os

print(os.path.exists(r"C:\Users\Binish\Desktop\LLM_FROM_SCRATCH\models\Llama-3.2-1B-Instruct\original\tokenizer.model"))
print(os.path.getsize(r"C:\Users\Binish\Desktop\LLM_FROM_SCRATCH\models\Llama-3.2-1B-Instruct\model.safetensors"))

True
2471645608


In [23]:
print(type(combined_weights))
print(len(combined_weights))
print(list(combined_weights.keys())[:10])

<class 'dict'>
146
['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight']


In [24]:

print(f"The device is : {device}")
torch.manual_seed(123)
tokenizer=Tokenizer(tokenizer_file)
llama3 = LLama3Model(LLAMA3_CONFIG_1B)
load_weights_into_llama(llama3, LLAMA3_CONFIG_1B, combined_weights)
llama3.to(device)
del combined_weights 
torch.manual_seed(123)
llama3.eval()

The device is : cuda
Model uses weight tying.


LLama3Model(
  (tok_emb): Embedding(128256, 2048)
  (trf_blocks): Sequential(
    (0): Transformer(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2048, out_features=2048, bias=False)
        (W_key): Linear(in_features=2048, out_features=512, bias=False)
        (W_value): Linear(in_features=2048, out_features=512, bias=False)
        (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
      (ff): FeedForward(
        (fc1): Linear(in_features=2048, out_features=8192, bias=False)
        (fc2): Linear(in_features=2048, out_features=8192, bias=False)
        (fc3): Linear(in_features=8192, out_features=2048, bias=False)
        (silu): SiLU()
      )
    )
    (1): Transformer(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2048, out_features=2048, bias=False)
        (W_key): Linear(in_features=2048, out_features=512, bias=False)
        (W_value): Linear(in_

In [25]:

STORE_PATH=r"C:\Users\Binish\Desktop\LLM_FROM_SCRATCH\vector_store"
from rag.rag_pipeline import build_prompt,RAG,VectorStore

rag=RAG(device=device)
vs=VectorStore.load(STORE_PATH)


c:\Users\Binish\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2606.02it/s]


In [ ]:
def chat(query):
    values,indices,retrieved_chunks=rag.retrieve(query,vs,k=5)
    prompt=build_prompt(query,retrieved_chunks)
    
    input_len=len(query)
    input_ids=text_to_token_ids(prompt,tokenizer).to(device)
    length=input_ids.shape[1]
    token_ids=generate_text(
        model=llama3,
        idx=input_ids,
        max_new_tokens=200,
        context_size=LLAMA3_CONFIG_1B["context_length"],
        top_k=1,
        temperature=0.,
        eos_id=tokenizer.eot_id
    )
    output=token_ids[:,length:]
    print(token_ids_to_text(output, tokenizer))

In [30]:
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich import box

console = Console()

title = Text("OSIRIS", style="bold cyan")
title.append("  |  Local Legal Intelligence", style="bold white")

console.print()

console.print(
    Panel.fit(
        "[bold cyan]Welcome! I am OSIRIS your legal law agent.[/bold cyan]\n\n"
        "[bold green]My key features include: [/bold green]\n\n"
        "->RAG Retrieval\n"
        "->Access to the Indian Legal Knowledge Base\n"
        "[yellow]Lets get started!\nType your legal question below.[/yellow]\n"
        "[dim]Type 'exit' to quit.[/dim]",
        title="[bold cyan]INITIALIZED AGENT[/bold cyan]",
        border_style="bright_cyan",
        box=box.DOUBLE,
        padding=(1, 4)
    )
)

console.print()

╔════════════════ INITIALIZED AGENT ════════════════╗
║                                                   ║
║    Welcome! I am OSIRIS your legal law agent.     ║
║                                                   ║
║    My key features include:                       ║
║                                                   ║
║    ->RAG Retrieval                                ║
║    ->Access to the Indian Legal Knowledge Base    ║
║    Lets get started!                              ║
║    Type your legal question below.                ║
║    Type 'exit' to quit.                           ║
║                                                   ║
╚═══════════════════════════════════════════════════╝

In [32]:
from rich.console import Console

console = Console()

while True:
    
    query = input()
    console.print(f"\n[bold cyan]User[/bold cyan]  > {query}", end="")

    if query.lower() == "exit":
        console.print("\n[bold red]OSIRIS shutting down...[/bold red]")
        break

    console.print("\n[bold green]OSIRIS[/bold green] >")
    answer = chat(query)
    console.print(answer)

User  >  Define fraud under the Indian Contract Act.

OSIRIS >



The provided legal documents do not contain sufficient information to answer this question.

The Indian Contract Act, 1872, defines fraud as:

"(1) the suggestion, as a fact, of that which is not true, by one who does not believe it to be true;
(2) the active concealment of a fact by one having knowledge or belief of the fact;
(3) a promise made without any intention of performing it;
(4) any other act fitted to deceive;
(5) any such act or omission as the law specially declares to be fraudulent."

Explanation.—Mere silence as to facts likely to affect the willingness of a person to enter into a contract is not fraud, unless the circumstances of the case are such that


None

User  > exit

OSIRIS shutting down...